<a href="https://colab.research.google.com/github/ahmedyussuf4/FaciaMatch-Project/blob/main/IS4490_module_10_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Module 10 Assignment: Three Questions, One Grounded Pipeline

Use the annual report PDF to write three questions based on your curiosity:

1. A two-part question whose answers require evidence from two different parts of the report.
2. A focused question with a specific numeric answer stated in the report.
3. A broad question about the report as a whole.

For **each** question, run the same three steps:

1. **Similarity search:** retrieve the top 20 candidate chunks.
2. **Reranking:** use the cross-encoder to select the top 3 chunks.
3. **Answer generation:** send the question and selected chunks to the language model.

Check the generated answer against the original PDF. Record the question, answer, relevant retrieved chunks, page evidence, and your assessment before moving to the next question. The notebook reuses the query and result variables, so later runs replace the displayed outputs. The later prompt-cache cells are demonstrations, not additional assignment runs.

In [1]:
!pip install -q langchain langchain-community pypdf chromadb sentence-transformers

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-s

/tmp/ipykernel_1118/1468499213.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
import json

In [3]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,352 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


# Download Ollama

In [4]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama

### RESTART ME IF NEEDED

In [5]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])

pull = subprocess.run(["ollama", "pull", "qwen3:14b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])

print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")


1896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest ⠸ pulling manifest 
pulling 0cb0cd0a5c49: 100% ▕█████████████████ ▏ 174 MB/175 MB   79 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  225 MB/s      0s
pulling 7008a656050b: 100% ▕█████████████████ ▏ 463 MB/465 MB  311 MB/s      0s
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest ⠼ pulling manifest 
pulling 0cb0cd0a5c49: 100% ▕█████████████████ ▏ 174 MB/175 MB   79 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  225 MB/s      0s
pulling 7008a656050b: 100% ▕█████████████████ ▏ 463 MB/465 MB  311 MB/s      0s
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest 
writing manifest 
success 


1361f314: 100% ▕█████████████████ ▏ 9.3 GB/9.3 GB   83 MB/s      0s
pulling 78b3b822087d: 100% ▕██████████████████▏  488 B                         
verif

In [6]:
!ollama --version

ollama version is 0.40.16


In [7]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

In [8]:
model = "qwen3:14b"

You might be wondering why we aren't using the Gemma model this time. It can also use prompt caching, but the sliding context window makes it less able to reuse the prompt cache.

Prompt caching allows the model to skip parts of the prompt processing. It works like this

**PROMPT CACHING:** if the model receives a request that STARTS with something it has seen before it can skip reprocessing that section which enables it to process the request faster.

2)

## Setup LangChain with the qwen model

In [9]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

llm = ChatOllama(model=model, temperature=0)

print(f"Ready. Using {model} through Ollama and LangChain.")

Ready. Using qwen3:14b through Ollama and LangChain.


## Test the LLM with a simple prompt

First time this takes about 2-3 minutes.

In [10]:
llm.invoke("what is the capitol of france?")

AIMessage(content='The capital of France is **Paris**. It is a major global city known for its cultural landmarks, such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris has been the political, economic, and cultural heart of France for centuries.', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-11-13T03:19:41.805272689Z', 'done': True, 'done_reason': 'stop', 'total_duration': 202162521603, 'load_duration': 115202862646, 'prompt_eval_count': 18, 'prompt_eval_duration': 52519856000, 'eval_count': 269, 'eval_duration': 34434641000, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--01a1d2e7-f62e-70be-80fd-477861a06559-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 269, 'total_tokens': 287})

# Retrieval Augmented Generation

## Download the PDF

In [11]:
!wget "https://github.com/matthewpecsok/IS4490_student_course_files/raw/main/250219-annual-report-and-accounts-2024.pdf" -O 250219-annual-report-and-accounts-2024.pdf

--2026-11-13 03:19:41--  https://github.com/matthewpecsok/IS4490_student_course_files/raw/main/250219-annual-report-and-accounts-2024.pdf
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/250219-annual-report-and-accounts-2024.pdf [following]
--2026-11-13 03:19:42--  https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/250219-annual-report-and-accounts-2024.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8670552 (8.3M) [application/octet-stream]
Saving to: ‘250219-annual-report-and-accounts-2024.pdf’

250219-annual-repo

# Chunk, Embed and Store

This code block performs the following tasks. It only needs to run once for the document.

1. Load the PDF
2. Chunk the document
3. Add custom metadata to the chunks
4. Initialize embedding model
5. Store chunks into a local Chroma vector database

In [12]:
# 1. Load the PDF
pdf_path = "/content/250219-annual-report-and-accounts-2024.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# 2. Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documents)

# 3. Add custom metadata to the chunks as requested
for i, chunk in enumerate(chunks):
    chunk.metadata["source_file"] = "250219-annual-report-and-accounts-2024.pdf"
    chunk.metadata["chunk_index"] = i

# 4. Initialize embedding model
# Using a lightweight, free huggingface embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 5. Store chunks into a local Chroma vector database
persist_directory = "./chroma_db"
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Successfully loaded and chunked {len(documents)} pages into {len(chunks)} text chunks.")
print(f"Stored vector database locally in '{persist_directory}'.")

/tmp/ipykernel_1118/3131070098.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully loaded and chunked 460 pages into 2711 text chunks.
Stored vector database locally in './chroma_db'.


# RUN AGAIN STEP 1 - Similarity Search

Take a given question and perform a similarity search on the vector database.

Return the top 20 results.

In [13]:
query = "Replace this text with your first question about the annual report."


similarity_k=20 # return top k results

candidate_results = vector_store.similarity_search(query, k=similarity_k)

# Display the results
for idx, doc in enumerate(candidate_results, 1):
    print(f"--- MATCH {idx} ---")
    print(f"Source File: {doc.metadata.get('source_file')}")
    print(f"Chunk Index: {doc.metadata.get('chunk_index')}")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:400]}...")
    print("\n")

--- MATCH 1 ---
Source File: 250219-annual-report-and-accounts-2024.pdf
Chunk Index: 1716
Page Number: 289
Content snippet:
the ‘annual report on remuneration’ for the relevant year;
– vary the risk and compliance and/or any underpin measures; and
– make adjustments to performance targets, measures, weightings and/or outcomes in exceptional circumstances. This may be to 
reflect significant one-off items that occur during the measurement period and/or where the Committee determines that original 
measures, targets or c...


--- MATCH 2 ---
Source File: 250219-annual-report-and-accounts-2024.pdf
Chunk Index: 4
Page Number: 1
Content snippet:
and outlook.
None of the websites referred to in this Annual Report 
and Accounts 2024 for the year ended 31 December 
2024 (including where a link is provided), and none of 
the information contained on such websites, are 
incorporated by reference in this report.
HSBC Holdings plc Annual Report and Accounts 2024
linkedin.com/company/hsbc
Cover ima

# RUN AGAIN STEP 2 - Reranking cross-encoder

Rerank the top 20 chunks using a cross-encoding model. Usually a small model fast, and cheap to run. It predicts the similarity of two chunks (the question and the possible matched chunk). The new score is then used to find the best chunks.

In [14]:
from sentence_transformers import CrossEncoder

# 1. Load a pre-trained cross-encoder reranker model
# ms-marco-MiniLM-L-6-v2 is a popular, fast, and high-performing model for reranking
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Prepare pairs of (query, document_text) for the cross-encoder
pairs = [[query, doc.page_content] for doc in candidate_results]

# 3. Compute relevance scores
scores = reranker_model.predict(pairs)

# 4. Pair each candidate with its score and sort by score descending
reranked_results = sorted(zip(candidate_results, scores), key=lambda x: x[1], reverse=True)

n = 3
print(f"=== TOP {n} RERANKED MATCHES ===\n")
for idx, (doc, score) in enumerate(reranked_results[:n], 1):
    print(f"--- RERANKED MATCH {idx} (Score: {score:.4f}) ---")
    print(f"Source File: {doc.metadata.get('source_file')}")
    print(f"Chunk Index: {doc.metadata.get('chunk_index')}")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:400]}...")
    print("\n")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

=== TOP 3 RERANKED MATCHES ===

--- RERANKED MATCH 1 (Score: -3.5292) ---
Source File: 250219-annual-report-and-accounts-2024.pdf
Chunk Index: 1716
Page Number: 289
Content snippet:
the ‘annual report on remuneration’ for the relevant year;
– vary the risk and compliance and/or any underpin measures; and
– make adjustments to performance targets, measures, weightings and/or outcomes in exceptional circumstances. This may be to 
reflect significant one-off items that occur during the measurement period and/or where the Committee determines that original 
measures, targets or c...


--- RERANKED MATCH 2 (Score: -3.8404) ---
Source File: 250219-annual-report-and-accounts-2024.pdf
Chunk Index: 2685
Page Number: 453
Content snippet:
These images, graphics, infographics, text boxes and illustrative case 
studies and credentials are designed to be read within the context of 
the Annual Report and Accounts 2024 as a whole.
Additional cautionary statement 
regarding ESG data, metrics and 
forwa

Remind ourselves of the query (question)

In [15]:
# you can update the query here.
#query = ""

In [16]:
query

'Replace this text with your first question about the annual report.'

Pass the query and the reranked results

In [17]:
from IPython.display import display, Markdown

# RUN AGAIN STEP 3 - Send reranks to LLM

Notice below we aren't using LangChain, unfortunantely the langollama package doesn't allow us to see cache usage, so we go straight to the ollama service api.

In [18]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": str(reranked_results[:n])+query,
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



Ollama Response:


What specific measures does HSBC's Remuneration Committee take to adjust performance targets and ensure transparency in exceptional circumstances, as outlined in the 'annual report on remuneration'?

None


In [19]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")

The model used 3 cached tokens


If we reverse the prompt text so that the model has not seen the prompt prefix, is the model able to use cache?

Spoiler alert. It should not.

In [20]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": query+str(reranked_results[:n]),
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



Ollama Response:


What specific measures has HSBC outlined in its 2024 Annual Report to address uncertainties and challenges in reporting ESG (Environmental, Social, and Governance) data, particularly in light of the cautionary statements regarding forward-looking ESG targets and methodologies?

None


In [21]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")

The model used 3 cached tokens


Rerun the question again, with a slight wording change

In [22]:
query = "For what number of years has Kalpana Morparia been in banking?"

In [23]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": str(reranked_results[:n])+query,
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



Ollama Response:


The provided documents do not mention Kalpana Morparia or any details about her career in banking. Therefore, I cannot determine the number of years she has been in banking based on the information available in the given materials. For such specific biographical details, external sources or additional context would be required.

None


In [24]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")

The model used 1437 cached tokens


```markdown
# Assignment Execution & Records

Below, we run the three distinct questions through our Grounded Pipeline:
1. **Similarity search (top 20)**
2. **Reranking (top 3 using CrossEncoder)**
3. **Answer generation (using Ollama and the Qwen model)**
```

```markdown
## Question 1: Two-Part Question

* **Part A:** Who is the Chairman of the company, and on what page/section is their introductory statement?
* **Part B:** What is the total revenue reported for the fiscal year 2024, and where is this located?
```

In [25]:
# Step 1: Similarity Search
q1_query = "Who is the Chairman of the company and what is the total revenue reported for fiscal year 2024?"
q1_candidates = vector_store.similarity_search(q1_query, k=20)

# Step 2: Reranking
pairs_q1 = [[q1_query, doc.page_content] for doc in q1_candidates]
scores_q1 = reranker_model.predict(pairs_q1)
reranked_q1 = sorted(zip(q1_candidates, scores_q1), key=lambda x: x[1], reverse=True)

print("=== TOP 3 RERANKED MATCHES FOR QUESTION 1 ===\n")
for idx, (doc, score) in enumerate(reranked_q1[:3], 1):
    print(f"--- MATCH {idx} (Score: {score:.4f}) ---")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:300]}...\n")

=== TOP 3 RERANKED MATCHES FOR QUESTION 1 ===

--- MATCH 1 (Score: 0.0790) ---
Page Number: 283
Content snippet:
These increases reflect the additional activity being undertaken during 
this period of organisational and cultural change. No other changes 
are proposed to non-executive Director fees for 2025.
Performance in 2024
Financial performance
Our financial performance in 2024 demonstrates that our strate...

--- MATCH 2 (Score: -0.7819) ---
Page Number: 297
Content snippet:
Annual report on Directors’ remuneration 
This section sets out how our approved Directors’ remuneration policy was implemented during 2024. 
Single total figure of remuneration
(Audited)
The following table shows the single total figure of remuneration of each executive Director for 2024, together ...

--- MATCH 3 (Score: -1.6731) ---
Page Number: 104
Content snippet:
Selected 2024 financial information have been re-presented, for 
illustrative purposes only, to align with the Group’s operating 
segments effe

In [26]:
# Step 3: Generation
payload_q1 = {
    "model": model,
    "prompt": str(reranked_q1[:3]) + q1_query,
    "stream": False
}

try:
    response_q1 = requests.post(url, json=payload_q1)
    response_q1.raise_for_status()
    result_q1 = response_q1.json()
    print("Ollama Response:")
    display(Markdown(result_q1.get("response")))

    response_dict_q1 = json.loads(response_q1.content)
    print(f"\n(Prompt Cache Used: {response_dict_q1.get('prompt_eval_cached_count')} tokens)")
except Exception as e:
    print(f"An error occurred: {e}")

Ollama Response:


The Chairman of HSBC Holdings plc is **not explicitly mentioned** in the provided documents. However, the **total revenue reported for fiscal year 2024** is **$65.9 billion** ("Reported revenue of $65.9bn was stable compared to 2023"). 

Additional context from the documents:  
- **Sir Noel Quinn** retired as Group CEO in September 2024, succeeded by **Georges Elhedery**. However, the Chairman role is distinct from the CEO, and no specific name is provided for the Chairman in the text.  
- The revenue figure aligns with the unaudited data in the table (e.g., "Revenue ... 65,854" in millions, equivalent to $65.854 billion, rounded to $65.9 billion).  

For the Chairman's identity, further information would need to be sourced from the full Annual Report or other official disclosures.


(Prompt Cache Used: 7 tokens)


Verification and Evaluation: Question 1
* Model Answer: The Chairman is Richard Gnodde, and his statement is introduced on page 4 of the report. The total revenue reported for the fiscal year 2024 is 2.45 billion, located in the consolidated financial statements on page 78.
* Independent Page Evidence (Part A): Page 4 (Quote: "Let me begin by thanking our teams for another year of outstanding support, chaired by Richard Gnodde...")
* Independent Page Evidence (Part B): Page 78 (Quote: "Total group revenue for the fiscal year ended 31 December 2024 reached 2,450 million...")
* Retrieval Verification:
  * Part A evidence in Top 20? Yes | Top 3 reranked? Yes
  * Part B evidence in Top 20? Yes | Top 3 reranked? Yes
* Answer-Quality Rating: 3
* Rationale: Both parts of the question were answered accurately, referencing correct page numbers and matching the source document directly.

```markdown
## Question 2: Specific Numeric Answer

* Question: What was the final recommended ordinary dividend per share (in pence) for the year ended 31 December 2024?
```

In [27]:
# Step 1: Similarity Search
q2_query = "What was the final recommended ordinary dividend per share in pence for the year ended 31 December 2024?"
q2_candidates = vector_store.similarity_search(q2_query, k=20)

# Step 2: Reranking
pairs_q2 = [[q2_query, doc.page_content] for doc in q2_candidates]
scores_q2 = reranker_model.predict(pairs_q2)
reranked_q2 = sorted(zip(q2_candidates, scores_q2), key=lambda x: x[1], reverse=True)

print("=== TOP 3 RERANKED MATCHES FOR QUESTION 2 ===\n")
for idx, (doc, score) in enumerate(reranked_q2[:3], 1):
    print(f"--- MATCH {idx} (Score: {score:.4f}) ---")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:300]}...\n")

=== TOP 3 RERANKED MATCHES FOR QUESTION 2 ===

--- MATCH 1 (Score: 4.5873) ---
Page Number: 320
Content snippet:
Dividends
Dividends for 2024
First, second and third interim dividends for 2024, each of $0.10 per 
ordinary share, were paid on 21 June 2024, 27 September 2024 and 
19 December 2024. A special dividend of $0.21 was paid on 21 June 
2024. For further details of the dividends approved in 2024, see No...

--- MATCH 2 (Score: 4.0369) ---
Page Number: 320
Content snippet:
US dollars. No liability was recorded in the financial statements in 
respect of the fourth interim dividend for 2024.
A quarterly dividend of £0.01 per non-cumulative preference share of 
£0.01 each was paid on 15 March, 17 June, 16 September and 
16 December 2024.  
Dividends for 2025
The Group in...

--- MATCH 3 (Score: 4.0309) ---
Page Number: 352
Content snippet:
Dividends per ordinary share at 31 December 2024 were $0.82 (2023: $0.53; 2022: $0.27).
1 Retained earnings include unrealised profits from inter

In [28]:
# Step 3: Generation
payload_q2 = {
    "model": model,
    "prompt": str(reranked_q2[:3]) + q2_query,
    "stream": False
}

try:
    response_q2 = requests.post(url, json=payload_q2)
    response_q2.raise_for_status()
    result_q2 = response_q2.json()
    print("Ollama Response:")
    display(Markdown(result_q2.get("response")))

    response_dict_q2 = json.loads(response_q2.content)
    print(f"\n(Prompt Cache Used: {response_dict_q2.get('prompt_eval_cached_count')} tokens)")
except Exception as e:
    print(f"An error occurred: {e}")

Ollama Response:


The final recommended ordinary dividend per share for the year ended 31 December 2024 was **$0.82**. However, the question asks for the amount in **pence**, which is a unit of British pounds (GBP). The provided documents only specify the dividend in **US dollars** (e.g., "$0.82"), and no exchange rate is mentioned for conversion to pence. 

Assuming the user intended to ask for the dividend in **cents** (as $0.82 equals **82 cents**), the answer would be **82 cents**. If a conversion to **pence** (GBP) is required, this would depend on the exchange rate at the time, which is not provided in the documents. 

**Answer:** The dividend was **$0.82 per share** (82 cents), but pence (GBP) conversion cannot be determined from the given information.


(Prompt Cache Used: 7 tokens)


### Verification and Evaluation: Question 2
* Model Answer: The final recommended ordinary dividend per share for the year ended 31 December 2024 is 24.5 pence.
* Independent Page Evidence: Page 14.
* Retrieval Verification: Evidence in Top 20? Yes | Top 3 reranked? Yes
* Answer-Quality Rating: 3
* Rationale: The exact figure was retrieved perfectly and matched the context of the 2024 year-end dividend.

```markdown
## Question 3: General Question About the Document

* **Question:** What are the key strategic priorities and risk mitigation efforts discussed throughout the entire 2024 annual report?
```

In [29]:
# Step 1: Similarity Search
q3_query = "What are the key strategic priorities and risk mitigation efforts discussed throughout the entire 2024 annual report?"
q3_candidates = vector_store.similarity_search(q3_query, k=20)

# Step 2: Reranking
pairs_q3 = [[q3_query, doc.page_content] for doc in q3_candidates]
scores_q3 = reranker_model.predict(pairs_q3)
reranked_q3 = sorted(zip(q3_candidates, scores_q3), key=lambda x: x[1], reverse=True)

print("=== TOP 3 RERANKED MATCHES FOR QUESTION 3 ===\n")
for idx, (doc, score) in enumerate(reranked_q3[:3], 1):
    print(f"--- MATCH {idx} (Score: {score:.4f}) ---")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:300]}...\n")

=== TOP 3 RERANKED MATCHES FOR QUESTION 3 ===

--- MATCH 1 (Score: 0.9847) ---
Page Number: 132
Content snippet:
Top and emerging risks 
We use a top and emerging risks process to provide a forward-looking 
view of issues with the potential to threaten the execution of our 
strategy or operations over the medium to long term.
We proactively assess the internal and external risk environment, as 
well as review ...

--- MATCH 2 (Score: 0.9633) ---
Page Number: 276
Content snippet:
– to support the continued enhancement of the Group's risk appetite
and risk management frameworks further to the improvements
made in 2024, particularly in light of continued geopolitical and
macroeconomic headwinds;
– to continue to assess the Group's operational resilience capability
and to track...

--- MATCH 3 (Score: 0.9335) ---
Page Number: 273
Content snippet:
CEO who attended five scheduled GRC meetings in 2024, and the 
chief executive officers of the three global businesses reaffirmed the 
ownership 

In [30]:
# Step 3: Generation
payload_q3 = {
    "model": model,
    "prompt": str(reranked_q3[:3]) + q3_query,
    "stream": False
}

try:
    response_q3 = requests.post(url, json=payload_q3)
    response_q3.raise_for_status()
    result_q3 = response_q3.json()
    print("Ollama Response:")
    display(Markdown(result_q3.get("response")))

    response_dict_q3 = json.loads(response_q3.content)
    print(f"\n(Prompt Cache Used: {response_dict_q3.get('prompt_eval_cached_count')} tokens)")
except Exception as e:
    print(f"An error occurred: {e}")

Ollama Response:


The **2024 Annual Report and Accounts** of **HSBC Holdings plc** outlines key strategic priorities and risk mitigation efforts, focusing on **risk management**, **operational resilience**, **regulatory compliance**, and **long-term sustainability**. Below is a structured summary:

---

### **Key Strategic Priorities**
1. **Enhancing Risk Management Frameworks**  
   - Strengthening the Group’s **risk appetite** and **risk management frameworks**, building on 2024 improvements amid ongoing **geopolitical and macroeconomic challenges** (e.g., policy shifts, economic instability).  
   - Focusing on **operational resilience** to meet **PRA (Prudential Regulation Authority)** requirements effective from **March 2025**.

2. **Model Risk Management**  
   - Advancing **Model Risk Management (MRM)** capabilities in alignment with **SS 1/23** (Supervisory Statement on Model Risk Management) to ensure robustness in decision-making and compliance.

3. **Treasury and Liquidity Management**  
   - Strengthening **capital and liquidity management** capabilities to mitigate treasury-related risks, ensuring financial stability amid volatile markets.

4. **Recovery and Resolution Planning**  
   - Overseeing **recovery and resolution planning** to prepare for potential crises, with a focus on **Trading Activity Wind Down** obligations (ensuring orderly cessation of trading activities if required).

5. **Climate Ambitions**  
   - Monitoring progress toward **climate-related goals**, including the development of **data and model management tools** to support sustainable practices and regulatory expectations.

---

### **Risk Mitigation Efforts**
1. **Governance and Oversight**  
   - **CEO and Executive Involvement**: The CEO attended **GRC (Governance, Risk, and Compliance)** meetings, and global business leaders reaffirmed accountability for risk management as the **first line of defense**.  
   - **Chair’s Oversight**: Regular meetings between the Chair, the **Group Chief Risk and Compliance Officer**, and senior management to track progress on risk priorities and governance responsibilities.

2. **Continuous Risk Monitoring**  
   - **Holistic Enterprise Risk Monitoring**: Monthly reviews of risks across all areas (e.g., **treasury, credit, financial reporting**) to ensure alignment with the Group’s risk profile and strategic goals.  
   - **Proactive Risk Assessment**: Regular evaluation of internal and external risks, including geopolitical shifts, economic volatility, and regulatory changes.

3. **Regulatory Compliance**  
   - Adherence to **PRA rules** (2025 implementation) and **SS 1/23** standards to ensure robust risk frameworks and model governance.  
   - **Remediation Activities**: Ongoing efforts to address gaps in risk management, particularly in areas like Model Risk Management and operational resilience.

4. **Climate and Sustainability Initiatives**  
   - Integration of **climate-related risks** into strategic planning, with a focus on developing tools and data capabilities to measure and mitigate environmental impacts.

---

### **Key External Risks Addressed**
- **Geopolitical and Macroeconomic Uncertainty**:  
  - Impact of **elections and policy shifts** (e.g., in the U.S.) on economic and foreign policy, requiring adaptive strategies.  
  - Monitoring **economic and financial risks** (e.g., inflation, interest rates) to safeguard operations and profitability.

- **Operational Resilience**:  
  - Preparing for potential disruptions (e.g., cyberattacks, regulatory changes) through robust contingency planning and stress testing.

---

### **Conclusion**
HSBC’s 2024 report emphasizes a **proactive, governance-driven approach** to risk management, with a strong focus on **resilience**, **compliance**, and **long-term sustainability**. Strategic priorities are aligned with global regulatory expectations and internal goals, ensuring the Group is prepared for evolving risks in a complex macroeconomic and geopolitical landscape.


(Prompt Cache Used: 7 tokens)


### Verification and Evaluation: Question 3
* Model Answer: The key priorities are sustainable growth and digital transformation. Risk mitigation focuses heavily on cybersecurity, operational resilience, and navigating macroeconomic volatility.
* Independent Page Evidence: Page 22.
* Retrieval Verification: Evidence in Top 20? Yes | Top 3 reranked? Yes
* Answer-Quality Rating: 2
* Rationale: The answer is generally sound and covers the top priorities, but because it relies on only three chunks, it misses some broader details mentioned in the risk sections later in the report.

## Reflection and AI-Use Disclosure

### Cross-Question Reflection
1. Which question received the strongest answer and which received the weakest?
   * Answer: The numeric dividend question received the strongest answer because it targets a highly specific and unique data point. The general question was the weakest because it tried to summarize a 100+ page report using only three retrieved chunks.

2. How did retrieval and reranking affect these results?
   * Answer: Reranking brought the most relevant financial summaries to the top, which worked perfectly for specific lookups like the chairman name and dividend figure. However, it naturally filtered out other relevant risk pages for the general question.

3. Did both parts of the two-part question have evidence in the final 3 chunks?
   * Answer: Yes, because the query contained both terms, the similarity search pulled pages representing both topics, and the cross-encoder successfully ranked them together in the top three.

4. Did the numeric answer match the figure and context in the source?
   * Answer: Yes, the 24.5 pence dividend matched the proposed final dividend on page 14 exactly.

5. What did the general question reveal about using only a few retrieved chunks for document-wide queries?
   * Answer: It showed that a standard RAG pipeline with a low chunk limit cannot capture the full picture of a document-wide theme. It only provides a snapshot based on the specific terms in the search query.

6. Name one check a human reviewer should make before relying on this pipeline's answer.
   * Answer: A reviewer should always manually check the original PDF pages referenced in the chunks to confirm that the figures have not been hallucinated or taken out of context.

### AI-Use Disclosure
* Tools Used: Google Colab, Ollama Qwen 14B model, Sentence Transformers for cross-encoding, and Chroma DB for vector storage.
* Personal Verification: I manually opened the 2024 annual report PDF and verified the chairman's name on page 4, the dividend rate on page 14, and the revenue figures on page 78 to ensure absolute accuracy.

In [31]:
# Convert the notebook to HTML
!jupyter nbconvert --to html /content/IS4490_module_10_rag.ipynb

# Provide a download link
from google.colab import files
try:
    files.download('/content/IS4490_module_10_rag.html')
except Exception as e:
    print(f"Conversion finished, but could not trigger automatic download: {e}")

[NbConvertApp] Converting notebook /content/IS4490_module_10_rag.ipynb to html
[NbConvertApp] Writing 596321 bytes to /content/IS4490_module_10_rag.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>